# Nano-Positioning System (NPS) with IMP.bff

This notebook demonstrates the NPS functionality of `IMP.bff`:

1. **Direct FRET efficiency** between two oriented dyes (`nps_direct_fret_efficiency`).
2. **Transfer anisotropy** (`nps_transfer_anisotropy`) — sensitized emission of an acceptor report.
3. **Multi-dye networks** — FRET efficiencies, transfer anisotropies, and a Gaussian
   log-likelihood over many dye pairs (`nps_network_*`), the dynamic-regime likelihood
   of the Fast-NPS approach (Muschielok & Michaelis, *Eur Biophys J* 2015).
4. **Convolved efficiencies** — orientation-grid coefficient tables (`eff_conv_coeff`)
   for the distance-distribution regime.
5. A thin **Bayesian structural restraint** (`NPSIsotropicFRETEfficiencyRestraint`)
   that scores live IMP particles with Monte Carlo movers.

Dye models: each dye carries a steady-state anisotropy $r$; the depolarization factor is
$q = \sqrt{r / r_0}$ with the fundamental anisotropy $r_0 = 0.4$ (see the Fast-NPS
references). The orientation-averaged $\langle\kappa^2\rangle$ and the efficiency follow
the depot factor algebra of the references above.

In [1]:
import math
import IMP
import IMP.bff as bff
import IMP.algebra

## 1. Direct FRET between two dyes

A `NPSDirectDye` places a dye with position $(x, y, z)$ and orientation
(polar angle $m$, azimuth $[0m\phi$) on its labeling site. The efficiency for the pair is

In [2]:
donor = bff.NPSDirectDye()
donor.x, donor.y, donor.z = 0.0, 0.0, 0.0
donor.m, donor.phi = 0.0, 0.0            # orientation: polar, azimuth
donor.steady_state_anisotropy = 0.3      # measured steady-state anisotropy

acceptor = bff.NPSDirectDye()
acceptor.x, acceptor.y, acceptor.z = 0.0, 0.0, 50.0   # 50 A along z
acceptor.m, acceptor.phi = 0.0, 0.0
acceptor.steady_state_anisotropy = 0.3

# R0 = 50 A for this donor/acceptor pair
eff = bff.nps_direct_fret_efficiency(donor, acceptor, 50.0)
print(f"E = {eff:.4f}")

E = 0.5573


With $\langle\kappa^2\rangle$ from the oriented dynamic model and $r = 50$ Å = $R_0$,
the isotropic value is $E = \langle\kappa^2\rangle R_0^6 / (\langle\kappa^2\rangle R_0^6 + r^6)$.

In [3]:
# polar angle sweep; m = cos(theta) in [-1, 1] is the dipole's
# projection on the inter-dye axis. E rises as the dyes align.
for m_val in (0.0, 0.5, 0.9):
    d = bff.NPSDirectDye()
    d.x, d.y, d.z = 0.0, 0.0, 0.0
    d.m, d.phi = m_val, 0.0
    d.steady_state_anisotropy = 0.3
    a = bff.NPSDirectDye()
    a.x, a.y, a.z = 0.0, 0.0, 50.0
    a.m, a.phi = 0.0, 0.0
    a.steady_state_anisotropy = 0.3
    print(f"polar angle {m_val:.3f}: E = {bff.nps_direct_fret_efficiency(d, a, 50.0):.4f}")

polar angle 0.000: E = 0.5573
polar angle 0.500: E = 0.5053
polar angle 0.900: E = 0.3283


## 2. Transfer anisotropy

`nps_transfer_anisotropy` returns the orientation-averaged transfer anisotropy for a pair —
the sensitized-emission anisotropy expected from the two dye transition dipoles.

In [4]:
ta = bff.nps_direct_transfer_anisotropy(donor, acceptor)
print('transfer anisotropy = %.4f' % ta)

transfer anisotropy = 0.3000


## 3. Multi-dye networks

The network API mirrors the Fast-NPS likelihood: a `config` table (one row per dye:
$x, y, z, m, [0m\phi$), a dye list with depolarization factors $q$, and measurements
(FRET efficiency and/or transfer anisotropy per dye pair). The likelihood is the sum of
Gaussian log densities over all active observables — the dynamic-regime core of
Fast-NPS `logLikelihood.cpp`.

`NPSNetworkDye.dep` is the depolarization factor $q = \sqrt{r / 0.4}$.

In [5]:
def make_config(rows):
    """rows: iterable of (x, y, z, m, phi) per dye."""
    return [list(r) for r in rows]

# Three dyes on a plausible structure
config = make_config([
    ( 0.0,  0.0,  0.0, 0.2, 1.0),
    (30.0, 10.0, 20.0, -0.3, 0.4),
    (-10.0, 40.0, 15.0, 0.5, -0.7),
])

# Dye optical parameters (steady-state anisotropies 0.25 / 0.3 / 0.2)
dyes = []
for r in (0.25, 0.3, 0.2):
    dye = bff.NPSNetworkDye()
    dye.dep = math.sqrt(r / 0.4)
    dyes.append(dye)

# FRET efficiency measured between dyes 0 and 1
eff = bff.nps_network_fret_efficiency(config, dyes, 0, 1, 55.0 ** 6)
print('network E(0,1) = %.4f' % eff)

ta01 = bff.nps_network_transfer_anisotropy(config, dyes, 0, 1)
print('network TA(0,1) = %.4f' % ta01)

network E(0,1) = 0.9239
network TA(0,1) = 0.0710


In [6]:
def make_measurement(dye1, dye2, fret=False, e_avg=0.0, e_err=1.0,
                     r_iso6=0.0, ta=False, r_t_avg=0.0, r_t_err=1.0):
    m = bff.NPSMeasurement()
    m.dye1, m.dye2 = dye1, dye2
    m.fret_active, m.e_avg, m.e_err = fret, e_avg, e_err
    m.r_iso6 = r_iso6                      # R0^6 for this pair
    m.ta_active, m.r_t_avg, m.r_t_err = ta, r_t_avg, r_t_err
    return m

measurements = [
    make_measurement(0, 1, fret=True, e_avg=0.60, e_err=0.08, r_iso6=55.0**6),
    make_measurement(0, 2, fret=True, e_avg=0.35, e_err=0.10, r_iso6=50.0**6),
    make_measurement(1, 2, fret=True, e_avg=0.50, e_err=0.12, r_iso6=60.0**6),
    make_measurement(0, 1, ta=True, r_t_avg=0.05, r_t_err=0.02),
]

ll = bff.nps_network_log_likelihood(config, dyes, measurements)
print('log likelihood = %.4f' % ll)

log likelihood = -15.8304


## 4. Convolved efficiencies (orientation grid)

When a dye samples a restricted orientational distribution that cannot be described by a
single depolarization factor, Fast-NPS pre-computes efficiencies on an orientation grid
(5 azimuthal $\times$ 5 polar slots, 25 rows per dye, 625 rows per pair) and
`nps_network_fret_efficiency` interpolates the table by the dyes' orientations
(row index $5\,\mathrm{int}(\frac{5}{3.142}\phi) + \mathrm{int}(2.5(\cos\theta+1))$,
the layout of `logLikelihood.cpp`). Passing `eff_conv_coeff` selects this regime.

In [7]:
# Orientation-grid coefficient table for a dye pair: 25 rows x 12
# coefficients. With every higher coefficient zero, the constant term
# is the efficiency itself — so a flat table means a constant E.
coeff = [[0.42] + [0.0] * 11 for i in range(25)]

# iso/iso dye pair: the table is indexed by the mean distance only
iso_dyes = []
for r in (0.25, 0.3):
    dye = bff.NPSNetworkDye()
    dye.dep = math.sqrt(r / 0.4)
    dye.iso = True
    dye.dist_conv = True
    iso_dyes.append(dye)

eff_conv = bff.nps_network_fret_efficiency(
    config, iso_dyes, 0, 1, 55.0 ** 6, coeff)
print('convolved E(0,1) = %.4f' % eff_conv)

convolved E(0,1) = 0.4200


## 5. Bayesian structural restraint on live particles

`NPSIsotropicFRETEfficiencyRestraint` scores a live IMP model: label sites are
`IMP::core::XYZR`-decorated particles, and the restraint adds
$-\log p(E_{\text{obs}} \mid \text{structure})$ to the model score. Monte Carlo movers
(e.g. `IMP::core::NormalMover`) sample positions; the restraint is total — unscorable
states return $+\infty$ so an MC step never throws.

In [8]:
model = IMP.Model()
p1 = IMP.Particle(model)
xyz1 = IMP.core.XYZR.setup_particle(p1, IMP.algebra.Sphere3D(IMP.algebra.Vector3D(0, 0, 0), 2.0))
p2 = IMP.Particle(model)
xyz2 = IMP.core.XYZR.setup_particle(p2, IMP.algebra.Sphere3D(IMP.algebra.Vector3D(0, 0, 48), 2.0))

# nuisance parameters: bias (log-scale FRET-rate correction) and
# log_r_iso (log of R0); give them finite initial values
bias = IMP.isd.Nuisance.setup_particle(
    IMP.Particle(model), 0.0)
log_r_iso = IMP.isd.Nuisance.setup_particle(
    IMP.Particle(model), math.log(55.0))

r = bff.NPSIsotropicFRETEfficiencyRestraint(
    model, xyz1, xyz2, bias, log_r_iso,
    0.55,        # observed efficiency
    0.08,        # observation error (sigma)
    "nps_demo")

print('restraint score at 48 A = %.4f' % r.evaluate(False))

xyz2.set_coordinates(IMP.algebra.Vector3D(0, 0, 62))
model.update()
print('restraint score at 62 A = %.4f' % r.evaluate(False))

restraint score at 48 A = 0.0032
restraint score at 62 A = 2.2555


## References

- Muschielok A, Michaelis J. *Application of the Nano-Positioning System to the
  thermodynamic analysis of fluorescently labeled macromolecules.* Eur Biophys J (2015).
- Muschielok A, et al. *A nano-positioning system for macromolecular structural studies.*
  Nat Methods (2008).
- Fast-NPS (CPC 2017) reference implementation — orientation-grid likelihood,
  distance-convolved coefficient tables.